# 02 - Baselines + MLflow + Fairlearn

Notebook na ordem recomendada para Fase 1:

1. Setup e carregamento
2. Tratamento minimo e split
3. Treino de baselines (DummyClassifier e Regressao Logistica)
4. Registro de experimentos no MLflow
5. Avaliacao de fairness com Fairlearn
6. Mitigacao de fairness e comparacao de trade-off

## 1) Setup e bibliotecas

Esta célula importa as bibliotecas usadas no experimento:
- manipulacao de dados (`pandas`, `numpy`)
- modelagem (`scikit-learn`)
- rastreabilidade (`mlflow`)
- fairness (`fairlearn`)

Resultado esperado: ambiente pronto para carregar dados, treinar baselines e medir desempenho/fairness.

In [16]:
import hashlib
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd

from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equalized_odds_difference,
    false_positive_rate,
    selection_rate,
    true_positive_rate,
    true_negative_rate,
    false_negative_rate,
)
from fairlearn.reductions import EqualizedOdds, ExponentiatedGradient
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


class IQRCapper(BaseEstimator, TransformerMixin):
    """Limita valores extremos ao intervalo [Q1 - factor*IQR, Q3 + factor*IQR].
    Deve ser fitado apenas no conjunto de treino para evitar data leakage.
    """

    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X = np.array(X, dtype=float)
        q1 = np.nanpercentile(X, 25, axis=0)
        q3 = np.nanpercentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lower_ = q1 - self.factor * iqr
        self.upper_ = q3 + self.factor * iqr
        return self

    def transform(self, X, y=None):
        X = np.array(X, dtype=float)
        return np.clip(X, self.lower_, self.upper_)


pd.set_option("display.max_columns", 200)

## 2) Carregamento do dataset

Esta célula lê o arquivo base de churn a partir de `../data/raw/telecom_churn_base_extended.csv`.

Resultado esperado:
- imprimir o shape bruto da base
- visualizar as primeiras linhas para validar colunas e tipos.

In [17]:
DATA_PATH = Path("../data/raw/telecom_churn_base_extended.csv")
df = pd.read_csv(DATA_PATH)
print("shape bruto:", df.shape)
display(df.head())

shape bruto: (51500, 73)


,customer_id,age,gender,region,plan_type,plan_price,is_promotional_plan,discount_amount,price_increase_last_3m,months_to_contract_end,contract_renewal_date,has_loyalty,months_to_loyalty_end,loyalty_end_date,internet_service,has_tv,has_fixed,tenure_months,network_outages_30d,avg_signal_quality,call_drop_rate,avg_internet_speed,service_failures_30d,installation_delay_days,repair_visits_90d,minutes_monthly,data_gb_monthly,sms_monthly,usage_delta_pct,days_since_last_usage,active_days_30d,night_usage_pct,roaming_usage,topup_frequency,avg_usage_last_3m,usage_trend_3m,payment_method,billing_type,late_payments_6m,invoice_shock_flag,avg_bill_last_6m,bill_variation_pct,collection_attempts_30d,monthly_charges,default_flag,days_past_due,support_calls_30d,support_calls_90d,complaints_30d,tickets_open_30d,last_complaint_reason,resolution_time_avg,first_call_resolution_flag,transferred_calls_count,last_contact_channel,app_login_30d,self_service_usage_30d,marketing_open_rate,campaign_response_flag,last_offer_accepted,days_since_last_interaction,portability_request_flag,sim_swap_count,competitor_offer_contact_flag,retention_offer_made,retention_offer_accepted,nps_score,nps_category,nps_promoter_flag,nps_detractor_flag,csat_score,churn,churn_probability
0,C000001,23,female,Sul,controle,38.73,0,0.00,0,12,2027-03-20,0,0,NaN,fiber,0,1,72,0,2.83,0.0025,32.65,0,0,0,349,0.12,315,0.379,57,3,39.7,0,1,0.12,0.251,boleto,digital,2,0,44.95,0.062,0,40.90,1,92,4,7,0,0,atraso_instalacao,0.72,0,0,telefone,54,3,39.0,0,0,102,0,0,0,0,0,10.0,promoter,1,0,5.0,1,0.9089
1,C000002,65,NaN,Nordeste,pos,55.93,1,16.97,0,28,2028-07-20,1,7,2026-10-20,none,0,1,92,0,4.30,0.0003,4.79,0,0,0,1998,1.13,17,-0.155,23,13,59.8,0,0,1.24,-0.157,debito,digital,0,0,39.76,-0.024,0,42.26,0,0,4,6,1,0,nenhum,1.08,0,0,telefone,43,11,45.8,1,0,52,0,1,0,0,0,6.0,detractor,0,1,4.0,1,0.2769
2,C000003,58,female,Sul,pos,61.65,0,0.00,0,0,2026-03-20,0,0,NaN,dsl,1,1,10,0,5.00,0.0029,0.30,0,5,0,521,10.46,267,-0.120,7,23,9.4,0,2,10.48,-0.342,card,digital,0,0,86.40,0.034,0,82.96,0,0,0,1,0,0,nenhum,0.92,1,0,app,31,0,51.2,1,0,50,0,0,0,0,0,10.0,promoter,1,0,5.0,0,0.0241
3,C000004,45,male,Sul,pos,63.73,0,0.00,0,22,2028-01-20,0,0,NaN,fiber,0,0,86,0,4.38,0.0016,12.29,0,0,1,2507,3.45,36,-0.388,24,26,20.6,0,1,4.13,-0.544,card,auto,0,0,79.99,-0.140,0,74.28,0,0,0,5,0,0,nenhum,0.12,0,0,app,13,26,23.4,0,0,173,0,0,0,0,0,10.0,promoter,1,0,5.0,0,0.0100
4,C000005,44,female,Sul,pos,64.58,0,0.00,1,30,2028-09-20,0,0,NaN,dsl,1,1,61,0,3.50,0.0014,25.65,1,0,0,1604,2.18,202,-0.223,44,10,5.0,0,2,2.79,-0.119,debito,paper,0,1,79.49,0.015,0,84.61,0,0,4,4,0,1,nenhum,0.43,0,0,telefone,38,27,44.7,0,0,21,0,0,0,0,0,7.0,passive,0,0,4.0,0,0.8085


## 3) Tratamento minimo dos dados

Esta etapa aplica limpeza e padronizacao simples para o baseline:
- remove duplicatas (incluindo por `customer_id`)
- padroniza `gender`, `plan_type` e `payment_method` (lowercase, categorias invalidas → `unknown`)
- cria `age_group` a partir de `age`
- aplica regras de negocio para corrigir valores fisicamente impossiveis (`age`, `monthly_charges`, `avg_signal_quality`, `call_drop_rate`, `nps_score`, `csat_score`)
- remove colunas com risco de leakage: `churn_probability`, `retention_offer_made`, `retention_offer_accepted`, `contract_renewal_date`, `loyalty_end_date`

Resultado esperado: base consistente, sem valores impossíveis e sem colunas que vazariam o alvo, pronta para split e pipeline estatistico.

In [18]:
# Tratamento minimo para baseline
df = df.drop_duplicates().copy()
if "customer_id" in df.columns:
    df = df.drop_duplicates(subset=["customer_id"], keep="first")

if "gender" in df.columns:
    df["gender"] = (
        df["gender"]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({"": np.nan, "nan": np.nan})
        .fillna("unknown")
    )

if "plan_type" in df.columns:
    df["plan_type"] = (
        df["plan_type"]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({"desconhecido": "unknown", "??": "unknown", "nan": "unknown"})
    )

if "payment_method" in df.columns:
    valid_payment = {"boleto", "debito", "card", "pix"}
    df["payment_method"] = df["payment_method"].apply(
        lambda x: x if str(x).strip().lower() in valid_payment else "unknown"
    )

if "age" in df.columns:
    bins = [0, 24, 34, 44, 54, 120]
    labels = ["<25", "25-34", "35-44", "45-54", "55+"]
    df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels, include_lowest=True)
    df["age_group"] = df["age_group"].astype(str).replace({"nan": "unknown"})
    # Idades fora do intervalo valido (18-100) sao fisicamente impossiveis
    df["age"] = df["age"].clip(18, 100)

# Regras de negocio: corrige valores fisicamente impossiveis antes do pipeline estatistico.
business_rules = {
    "monthly_charges":    (0, None),    # cobranca nao pode ser negativa
    "avg_signal_quality": (0, 5),       # escala definida pelo sistema: 0 a 5
    "call_drop_rate":     (0, 1),       # taxa: 0% a 100%
    "nps_score":          (0, 10),      # escala NPS: 0 a 10
    "csat_score":         (1, 5),       # escala CSAT: 1 a 5
}
for col, (low, high) in business_rules.items():
    if col in df.columns:
        df[col] = df[col].clip(lower=low, upper=high)

# Colunas removidas por risco de leakage:
# - customer_id: identificador sem valor preditivo
# - churn_probability: predicao direta do alvo (correlacao 0.69)
# - retention_offer_made / retention_offer_accepted: acoes tomadas apos o evento de churn
# - contract_renewal_date / loyalty_end_date: marcadores temporais que revelam o timing do churn
LEAKAGE_COLS = [
    "customer_id",
    "churn_probability",
    "retention_offer_made",
    "retention_offer_accepted",
    "contract_renewal_date",
    "loyalty_end_date",
]
df = df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns])

print("shape tratado:", df.shape)
display(df[["gender", "age_group", "plan_type", "nps_score", "csat_score", "churn"]].head())

shape tratado: (49049, 68)


,gender,age_group,plan_type,nps_score,csat_score,churn
0,female,<25,controle,10.0,5.0,1
1,unknown,55+,pos,6.0,4.0,1
2,female,55+,pos,10.0,5.0,0
3,male,45-54,pos,10.0,5.0,0
4,female,35-44,pos,7.0,4.0,0


## 4) Definicao de alvo, split e preprocessamento

Aqui definimos:
- target (`churn`) e 4 atributos sensiveis para analise de fairness:
  - `gender` e `age_group` — protecao demografica (LGPD)
  - `region` — equidade geografica (Anatel)
  - `plan_type` — proxy de renda (clientes pre-pago tendem a menor poder aquisitivo)
- split estratificado treino/teste (80/20)
- pipeline de preprocessamento:
  - **numericas**: imputacao pela mediana → StandardScaler (necessario para regularizacao L1/L2)
  - **categoricas**: imputacao pela moda → OneHotEncoder

Resultado esperado: `X_train`, `X_test`, `y_train`, `y_test` e `preprocess` prontos para treino.

In [19]:
target_col = "churn"

X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

sensitive_gender  = df["gender"].astype(str)    if "gender"    in df.columns else pd.Series(["unknown"] * len(df), index=df.index)
sensitive_age     = df["age_group"].astype(str)  if "age_group" in df.columns else pd.Series(["unknown"] * len(df), index=df.index)
sensitive_region  = df["region"].astype(str)     if "region"    in df.columns else pd.Series(["unknown"] * len(df), index=df.index)
sensitive_plan    = df["plan_type"].astype(str)  if "plan_type" in df.columns else pd.Series(["unknown"] * len(df), index=df.index)

(
    X_train, X_test,
    y_train, y_test,
    s_gender_train, s_gender_test,
    s_age_train,    s_age_test,
    s_region_train, s_region_test,
    s_plan_train,   s_plan_test,
) = train_test_split(
    X, y,
    sensitive_gender,
    sensitive_age,
    sensitive_region,
    sensitive_plan,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[
                ("imp",    SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),   # obrigatorio antes da regularizacao L1/L2
            ]),
            num_cols,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imp", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat_cols,
        ),
    ]
)

print("Treino/Teste:", X_train.shape, X_test.shape)

Treino/Teste: (39239, 67) (9810, 67)


## 5) Treino dos baselines e logging no MLflow

Esta etapa:
- define funcoes utilitarias (versao do dataset e ROC-AUC seguro)
- treina dois baselines (`dummy_stratified` e `log_reg`)
- calcula metricas de classificacao
- registra parametros e metricas no experimento `churn-baselines` no MLflow

Resultado esperado: tabela `perf_df` com comparacao de desempenho e identificacao do melhor baseline.

In [20]:
def compute_dataset_version(path: Path) -> str:
    data_bytes = path.read_bytes()
    digest = hashlib.sha256(data_bytes).hexdigest()[:12]
    return f"sha256:{digest}"


def safe_roc_auc(y_true, y_score) -> float:
    try:
        return float(roc_auc_score(y_true, y_score))
    except ValueError:
        return float("nan")


def compute_business_metrics(y_true, y_pred, value_retained: float, action_cost: float) -> dict:
    y_true_np = np.asarray(y_true).astype(int)
    y_pred_np = np.asarray(y_pred).astype(int)
    tp = int(((y_true_np == 1) & (y_pred_np == 1)).sum())
    fp = int(((y_true_np == 0) & (y_pred_np == 1)).sum())
    contacted = int((y_pred_np == 1).sum())
    n = int(len(y_true_np))
    gross_value = float(tp * value_retained)
    action_total_cost = float((tp + fp) * action_cost)
    net_value = float(gross_value - action_total_cost)
    value_per_customer = float(net_value / n) if n > 0 else float("nan")
    return {
        "tp": float(tp), "fp": float(fp),
        "clientes_abordados": float(contacted),
        "valor_bruto": gross_value,
        "custo_total_acao": action_total_cost,
        "valor_liquido": net_value,
        "valor_por_cliente": value_per_customer,
    }


LOG_REG_KWARGS = {"max_iter": 2000, "solver": "liblinear", "random_state": 42}
BUSINESS_VALUE_RETAINED = 500.0
BUSINESS_ACTION_COST = 50.0

dataset_version = compute_dataset_version(DATA_PATH)
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("churn-baselines")

models = {
    "dummy_stratified": DummyClassifier(strategy="stratified", random_state=42),
    "log_reg": LogisticRegression(**LOG_REG_KWARGS),
}

trained = {}
pred_store = {}
rows = []
train_rows = []

for name, estimator in models.items():
    pipe = Pipeline(steps=[("prep", preprocess), ("model", estimator)])
    pipe.fit(X_train, y_train)

    y_pred  = pipe.predict(X_test).astype(int)
    y_score = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else y_pred.astype(float)

    y_pred_train  = pipe.predict(X_train).astype(int)
    y_score_train = pipe.predict_proba(X_train)[:, 1] if hasattr(pipe, "predict_proba") else y_pred_train.astype(float)

    business_row = compute_business_metrics(y_test, y_pred, BUSINESS_VALUE_RETAINED, BUSINESS_ACTION_COST)

    metrics_row = {
        "model": name,
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "f1": float(f1_score(y_test, y_pred)),
        "roc_auc": safe_roc_auc(y_test, y_score),
        "pr_auc": float(average_precision_score(y_test, y_score)),
        "positive_rate": float(y_pred.mean()),
        **business_row,
    }
    rows.append(metrics_row)

    train_rows.append({
        "model":          name,
        "accuracy_train": float(accuracy_score(y_train, y_pred_train)),
        "f1_train":       float(f1_score(y_train, y_pred_train)),
        "roc_auc_train":  safe_roc_auc(y_train, y_score_train),
    })

    with mlflow.start_run(run_name=name):
        mlflow.log_params({
            "model": name, "test_size": 0.2, "random_state": 42,
            "dataset_version": dataset_version,
            "business_value_retained": BUSINESS_VALUE_RETAINED,
            "business_action_cost": BUSINESS_ACTION_COST,
        })
        mlflow.log_metrics({k: v for k, v in metrics_row.items() if k != "model" and pd.notna(v)})

    trained[name]    = pipe
    pred_store[name] = {"y_pred": y_pred, "y_score": y_score}

perf_df = pd.DataFrame(rows).sort_values("roc_auc", ascending=False, na_position="last").reset_index(drop=True)
display(perf_df)

best_name  = perf_df.loc[0, "model"]
best_model = trained[best_name]
print("Melhor baseline por ROC-AUC:", best_name)
print("Dataset version:", dataset_version)
print("Metrica de negocio: valor_liquido = TP * V_RETIDO - (TP + FP) * C_ACAO",
      f"| V_RETIDO={BUSINESS_VALUE_RETAINED:.2f} | C_ACAO={BUSINESS_ACTION_COST:.2f}")

# --- overfit_df: comparacao treino vs teste ---
train_df = pd.DataFrame(train_rows)
test_metrics = perf_df[["model", "accuracy", "f1", "roc_auc"]].rename(columns={
    "accuracy": "accuracy_test", "f1": "f1_test", "roc_auc": "roc_auc_test"
})
overfit_df = train_df.merge(test_metrics, on="model")

print("\nComparacao treino vs teste:")
display(overfit_df)

# --- matrizes de confusao ---
print("\nMatrizes de confusao (conjunto de teste):\n")
for name, payload in pred_store.items():
    cm = confusion_matrix(y_test, payload["y_pred"])
    cm_df = pd.DataFrame(
        cm,
        index=["Real: nao churn (0)", "Real: churn (1)"],
        columns=["Pred: nao churn (0)", "Pred: churn (1)"],
    )
    print(f"{name}:\n")
    display(cm_df)


,model,accuracy,f1,roc_auc,pr_auc,positive_rate,tp,fp,clientes_abordados,valor_bruto,custo_total_acao,valor_liquido,valor_por_cliente
0,log_reg,0.806932,0.742523,0.878335,0.848021,0.356167,2731.0,763.0,3494.0,1365500.0,174700.0,1190800.0,121.386340
1,dummy_stratified,0.529358,0.393697,0.504595,0.395905,0.382569,1499.0,2254.0,3753.0,749500.0,187650.0,561850.0,57.273191


Melhor baseline por ROC-AUC: log_reg
Dataset version: sha256:3a1b88dbc248
Metrica de negocio: valor_liquido = TP * V_RETIDO - (TP + FP) * C_ACAO | V_RETIDO=500.00 | C_ACAO=50.00

Comparacao treino vs teste:


,model,accuracy_train,f1_train,roc_auc_train,accuracy_test,f1_test,roc_auc_test
0,dummy_stratified,0.518999,0.388558,0.496065,0.529358,0.393697,0.504595
1,log_reg,0.811820,0.749100,0.879635,0.806932,0.742523,0.878335



Matrizes de confusao (conjunto de teste):

dummy_stratified:



,Pred: nao churn (0),Pred: churn (1)
Real: nao churn (0),3694,2254
Real: churn (1),2363,1499


log_reg:



,Pred: nao churn (0),Pred: churn (1)
Real: nao churn (0),5185,763
Real: churn (1),1131,2731


## 5a) Analise de overfitting: treino vs teste

Compara as metricas de treino e teste para diagnosticar overfitting, conforme ensinado na Aula 01.

- **delta = metrica_treino - metrica_teste**
- Delta proximo de zero: modelo generaliza bem
- Delta grande (> 0.05): modelo memorizou o treino — considere mais regularizacao


In [21]:
# ── 5a) Analise de overfitting: treino vs teste ────────────────────────────

OVERFIT_THRESHOLD = 0.05  # delta acima disso e sinal de atencao

train_cols = [c for c in overfit_df.columns if c.endswith('_train')]
metrics = [c.replace('_train', '') for c in train_cols]

ov = overfit_df.copy()
for m in metrics:
    ov[f'delta_{m}'] = (ov[f'{m}_train'] - ov[f'{m}_test']).round(4)

if 'delta_roc_auc' in ov.columns:
    ov['flag_overfit'] = ov['delta_roc_auc'].apply(
        lambda d: 'atencao' if d > OVERFIT_THRESHOLD else 'OK')

delta_cols = ['model'] + [c for c in ov.columns if c.startswith('delta_')] + ['flag_overfit']
delta_cols = [c for c in delta_cols if c in ov.columns]
print('\u2550' * 60)
print('  5a) Overfitting: treino vs teste')
print(f'   Limiar de atencao: delta_roc_auc > {OVERFIT_THRESHOLD}')
print('\u2550' * 60)

styled = (
    ov[delta_cols].style
    .format({c: '{:.4f}' for c in delta_cols if c not in ('model', 'flag_overfit')})
    .applymap(lambda v: 'background-color: #ffe0e0' if v == 'atencao' else '', subset=['flag_overfit'])
)
display(styled)

print('Interpretacao:')
for _, row in ov.iterrows():
    m = row.get('model', '')
    d = row.get('delta_roc_auc', float('nan'))
    flag = row.get('flag_overfit', 'OK')
    if flag == 'atencao':
        print(f'  [{m}] delta_roc_auc={d:.4f} -> ATENCAO: possivel overfitting')
    else:
        print(f'  [{m}] delta_roc_auc={d:.4f} -> generalizacao adequada')


════════════════════════════════════════════════════════════
  5a) Overfitting: treino vs teste
   Limiar de atencao: delta_roc_auc > 0.05
════════════════════════════════════════════════════════════


,model,delta_accuracy,delta_f1,delta_roc_auc,flag_overfit
0,dummy_stratified,-0.0104,-0.0051,-0.0085,OK
1,log_reg,0.0049,0.0066,0.0013,OK


Interpretacao:
  [dummy_stratified] delta_roc_auc=-0.0085 -> generalizacao adequada
  [log_reg] delta_roc_auc=0.0013 -> generalizacao adequada


## 5b) Tuning de regularizacao (GridSearchCV) e threshold

O baseline logistico usa C=1 (padrao). Esta etapa:
- aplica **GridSearchCV com StratifiedKFold (5 folds)** para encontrar o melhor C (Aula 01 + Aula 02)
- exibe os resultados de validacao cruzada por valor de C
- avalia diferentes **thresholds de decisao** para maximizar a metrica de negocio

Resultado esperado: melhor C, pipeline tunado e threshold otimizado.


In [22]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# --- GridSearchCV: tuning do parametro C com validacao cruzada estratificada ---
param_grid = {'model__C': [0.001, 0.01, 0.1, 1, 10, 100]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_grid = Pipeline(steps=[
    ('pre',   preprocess),
    ('model', LogisticRegression(penalty='l2', solver='liblinear',
                                  class_weight='balanced', max_iter=2000,
                                  random_state=42)),
])

gs = GridSearchCV(
    pipe_grid, param_grid,
    scoring='roc_auc', cv=cv,
    refit=True, n_jobs=-1, return_train_score=True,
)
gs.fit(X_train, y_train)

best_C = gs.best_params_['model__C']
best_pipe = gs.best_estimator_

cv_results = pd.DataFrame(gs.cv_results_)
cv_summary = pd.DataFrame({
    'C':                cv_results['param_model__C'].astype(float),
    'roc_auc_treino_cv': cv_results['mean_train_score'].round(6),
    'roc_auc_val_cv':    cv_results['mean_test_score'].round(6),
    'std_val_cv':        cv_results['std_test_score'].round(6),
})

print(f'Melhor C encontrado: {best_C}')
print(f'ROC-AUC medio (validacao cruzada): {gs.best_score_:.4f}')
print('\nResultados GridSearchCV por valor de C:')
print(cv_summary.to_string(index=False))


Melhor C encontrado: 0.1
ROC-AUC medio (validacao cruzada): 0.8782

Resultados GridSearchCV por valor de C:
      C  roc_auc_treino_cv  roc_auc_val_cv  std_val_cv
  0.001           0.875822        0.874666    0.003301
  0.010           0.879353        0.877895    0.003263
  0.100           0.879770        0.878167    0.003337
  1.000           0.879796        0.878163    0.003370
 10.000           0.879802        0.878153    0.003385
100.000           0.879805        0.878146    0.003395


## 5c) Comparacao de penalizacoes: L1 vs L2 vs Elastic Net

Testa tres tipos de regularizacao da Regressao Logistica via GridSearchCV com 5-fold estratificado, conforme ensinado na Aula 02.

| Penalizacao | Solver | Parametros buscados | Comportamento |
|-------------|--------|---------------------|---------------|
| L2 (Ridge) | liblinear | C | Encolhe coeficientes, mantem todas as features |
| L1 (Lasso) | liblinear | C | Encolhe e zera coeficientes — selecao implicita |
| ElasticNet | saga | C, l1_ratio | Hibrido L1+L2, mais flexivel |

**delta_overfit** = roc_auc_treino - roc_auc_teste: quanto menor, melhor a generalizacao.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score

C_grid = [0.001, 0.01, 0.1, 1, 10, 100]
cv_pen = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Otimização: solvers e tolerâncias para máxima velocidade e menos warnings
configs_pen = {
    'L2 (Ridge)': dict(penalty='l2', solver='lbfgs', max_iter=3000, tol=1e-2, n_jobs=-1),
    'L1 (Lasso)': dict(penalty='l1', solver='liblinear', max_iter=3000, tol=1e-2),
    'ElasticNet': dict(penalty='elasticnet', solver='saga', max_iter=3000, l1_ratio=0.5, tol=1e-2, n_jobs=-1),
}
param_grids_pen = {
    'L2 (Ridge)': {'model__C': C_grid},
    'L1 (Lasso)': {'model__C': C_grid},
    'ElasticNet': {'model__C': C_grid, 'model__l1_ratio': [0.2, 0.5, 0.8]},
}

pen_rows = []
best_penalty_pipes = {}

for pen_name, cfg in configs_pen.items():
    pipe_pen = Pipeline(steps=[
        ('pre',   preprocess),
        ('model', LogisticRegression(**cfg, class_weight='balanced', random_state=42)),
    ])
    gs_pen = GridSearchCV(
        pipe_pen, param_grids_pen[pen_name],
        scoring='roc_auc', cv=cv_pen,
        refit=True, n_jobs=-1, return_train_score=True,
    )
    gs_pen.fit(X_train, y_train)
    best_penalty_pipes[pen_name] = gs_pen.best_estimator_

    y_pred_pen   = gs_pen.best_estimator_.predict(X_test)
    y_prob_pen   = gs_pen.best_estimator_.predict_proba(X_test)[:, 1]
    y_pred_train = gs_pen.best_estimator_.predict(X_train)
    y_prob_train = gs_pen.best_estimator_.predict_proba(X_train)[:, 1]

    tp_pen = int(((y_pred_pen == 1) & (y_test == 1)).sum())
    fp_pen = int(((y_pred_pen == 1) & (y_test == 0)).sum())
    abord  = tp_pen + fp_pen
    vl_pen = tp_pen * 500 - abord * 50

    pen_rows.append({
        'penalizacao':     pen_name,
        'melhor_C':        gs_pen.best_params_['model__C'],
        'roc_auc_cv':      round(gs_pen.best_score_, 4),
        'roc_auc_treino':  round(roc_auc_score(y_train, y_prob_train), 4),
        'roc_auc_teste':   round(roc_auc_score(y_test,  y_prob_pen),   4),
        'delta_overfit':   round(roc_auc_score(y_train, y_prob_train) - roc_auc_score(y_test, y_prob_pen), 4),
        'accuracy_teste':  round(accuracy_score(y_test, y_pred_pen),   4),
        'f1_teste':        round(f1_score(y_test,        y_pred_pen),   4),
        'pr_auc_teste':    round(average_precision_score(y_test, y_prob_pen), 4),
        'valor_liquido':   vl_pen,
    })
    print(f'  {pen_name:<12} melhor_C={gs_pen.best_params_["model__C"]}  '
          f'roc_cv={gs_pen.best_score_:.4f}  roc_test={roc_auc_score(y_test, y_prob_pen):.4f}')

penalty_df = pd.DataFrame(pen_rows).set_index('penalizacao')
print()
display(
    penalty_df.style
    .format({'roc_auc_cv': '{:.4f}', 'roc_auc_treino': '{:.4f}',
             'roc_auc_teste': '{:.4f}', 'delta_overfit': '{:.4f}',
             'accuracy_teste': '{:.4f}', 'f1_teste': '{:.4f}',
             'pr_auc_teste': '{:.4f}', 'valor_liquido': '{:,.0f}'})
    .highlight_max(subset=['roc_auc_teste', 'f1_teste', 'pr_auc_teste', 'valor_liquido'], color='#d4edda')
    .highlight_min(subset=['delta_overfit'], color='#d4edda')
)


  L2 (Ridge)   melhor_C=100  roc_cv=0.8782  roc_test=0.8783
  L1 (Lasso)   melhor_C=0.1  roc_cv=0.8783  roc_test=0.8783


/root/mlet-grupo4-tech-challenge/.venv-2/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/root/mlet-grupo4-tech-challenge/.venv-2/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/root/mlet-grupo4-tech-challenge/.venv-2/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/root/mlet-grupo4-tech-challenge/.venv-2/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/root/mlet-grupo4-tech-challenge/.venv-2/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not c

KeyboardInterrupt: 

## 6) Avaliacao de fairness por grupo sensivel

Com as predicoes dos baselines, esta célula calcula indicadores de fairness por:
- `gender`
- `age_group`

Metricas principais:
- `dp_diff` (demographic parity)
- `eo_diff` (equalized odds)
- gaps de taxa de selecao, TPR e FPR

Resultado esperado: `fairness_df` comparando disparidades entre grupos para cada modelo.

In [ ]:
fair_rows = []

for model_name, payload in pred_store.items():
    y_pred_model = payload["y_pred"]

    for sensitive_name, sensitive_values in [
        ("gender", s_gender_test),
        ("age_group", s_age_test),
    ]:
        mf = MetricFrame(
            metrics={
                "selection_rate": selection_rate,
                "tpr": true_positive_rate,
                "tnr": true_negative_rate,
                "fpr": false_positive_rate,
                "fnr": false_negative_rate,
            },
            y_true=y_test,
            y_pred=y_pred_model,
            sensitive_features=sensitive_values,
        )

        fair_rows.append(
            {
                "model": model_name,
                "sensitive_feature": sensitive_name,
                "dp_diff": float(
                    demographic_parity_difference(
                        y_test, y_pred_model, sensitive_features=sensitive_values
                    )
                ),
                "eo_diff": float(
                    equalized_odds_difference(
                        y_test, y_pred_model, sensitive_features=sensitive_values
                    )
                ),
                "selection_rate_gap": float(mf.difference(method="between_groups")["selection_rate"]),
                "tpr_gap": float(mf.difference(method="between_groups")["tpr"]),
                "fpr_gap": float(mf.difference(method="between_groups")["fpr"]),
            }
        )

fairness_df = pd.DataFrame(fair_rows).sort_values(["model", "sensitive_feature"]).reset_index(drop=True)
display(fairness_df)

,model,sensitive_feature,dp_diff,eo_diff,selection_rate_gap,tpr_gap,fpr_gap
0,dummy_stratified,age_group,0.031805,0.067957,0.031805,0.048495,0.067957
1,dummy_stratified,gender,0.023419,0.036503,0.023419,0.036503,0.025367
2,log_reg,age_group,0.034179,0.135786,0.034179,0.135786,0.039362
3,log_reg,gender,0.044792,0.066339,0.044792,0.052598,0.066339


## 7) Mitigacao de fairness com Equalized Odds

Aqui aplicamos mitigacao no baseline logistico usando `ExponentiatedGradient` com restricao `EqualizedOdds`.

Decisoes de implementacao:
- usa uma amostra estratificada de treino para reduzir tempo de execucao
- mede tempo de treinamento da mitigacao
- compara metricas de qualidade e fairness no conjunto de teste
- registra o experimento mitigado no MLflow

Resultado esperado: tabela com metricas do modelo mitigado para comparar trade-off entre performance e equidade.

In [ ]:
# Mitigacao de fairness no modelo logistico com EqualizedOdds (versao mais rapida)
from time import perf_counter

# Mitigacao em amostra estratificada para reduzir tempo de execucao.
MITIGATION_SAMPLE_SIZE = 15000
if len(X_train) > MITIGATION_SAMPLE_SIZE:
    X_mit, _, y_mit, _, s_gender_mit, _ = train_test_split(
        X_train,
        y_train,
        s_gender_train,
        train_size=MITIGATION_SAMPLE_SIZE,
        stratify=y_train,
        random_state=42,
    )
else:
    X_mit, y_mit, s_gender_mit = X_train, y_train, s_gender_train

X_mit_t = preprocess.fit_transform(X_mit)
X_test_t = preprocess.transform(X_test)

mitigation_estimator = LogisticRegression(
    max_iter=200000,
    solver="liblinear",
    random_state=42,
)

mitigator = ExponentiatedGradient(
    estimator=mitigation_estimator,
    constraints=EqualizedOdds(),
    eps=0.02,
    max_iter=15,
)

t0 = perf_counter()
mitigator.fit(X_mit_t, y_mit, sensitive_features=s_gender_mit)
elapsed = perf_counter() - t0
print(f"Tempo de mitigacao: {elapsed:.1f}s (n_treino_mitig={len(X_mit):,})")

y_pred_mitig = mitigator.predict(X_test_t).astype(int)

business_mitig = compute_business_metrics(
    y_true=y_test,
    y_pred=y_pred_mitig,
    value_retained=BUSINESS_VALUE_RETAINED,
    action_cost=BUSINESS_ACTION_COST,
)

mitig_metrics = {
    "model": "log_reg_mitigated_equalized_odds",
    "accuracy": float(accuracy_score(y_test, y_pred_mitig)),
    "f1": float(f1_score(y_test, y_pred_mitig)),
    "positive_rate": float(y_pred_mitig.mean()),
    "dp_diff_gender": float(
        demographic_parity_difference(y_test, y_pred_mitig, sensitive_features=s_gender_test)
    ),
    "eo_diff_gender": float(
        equalized_odds_difference(y_test, y_pred_mitig, sensitive_features=s_gender_test)
    ),
    **business_mitig,
}

with mlflow.start_run(run_name="log_reg_mitigated_equalized_odds"):
    mlflow.log_params(
        {
            "model": "log_reg_mitigated_equalized_odds",
            "base_model": "log_reg",
            "constraint": "EqualizedOdds",
            "sensitive_feature": "gender",
            "dataset_path": str(DATA_PATH),
            "dataset_version": dataset_version,
            "mitigation_sample_size": int(len(X_mit)),
            "mitigation_eps": 0.02,
            "mitigation_max_iter": 15,
            "business_value_retained": BUSINESS_VALUE_RETAINED,
            "business_action_cost": BUSINESS_ACTION_COST,
            "business_formula": "valor_liquido = TP * V_RETIDO - (TP + FP) * C_ACAO",
        }
    )
    mlflow.log_metrics({k: v for k, v in mitig_metrics.items() if k != "model" and pd.notna(v)})

display(pd.DataFrame([mitig_metrics]))


Tempo de mitigacao: 42.2s (n_treino_mitig=15,000)


,model,accuracy,f1,positive_rate,dp_diff_gender,eo_diff_gender,tp,fp,clientes_abordados,valor_bruto,custo_total_acao,valor_liquido,valor_por_cliente
0,log_reg_mitigated_equalized_odds,0.803262,0.736986,0.354332,0.052206,0.068858,2704.0,772.0,3476.0,1352000.0,173800.0,1178200.0,120.101937


## 8) Tabela comparativa final dos modelos

Consolida todos os modelos treinados neste notebook em uma unica visao para orientar a decisao operacional.

| Modelo | Descricao |
|--------|-----------|
| `dummy_stratified` | Baseline aleatorio — piso minimo de performance |
| `log_reg` | Regressao Logistica com C=1 (padrao) |
| Melhor penalizacao | Melhor entre L1/L2/ElasticNet por ROC-AUC |
| `log_reg_mitigated_equalized_odds` | Modelo mitigado por fairness |


In [ ]:
# ── 8) Tabela comparativa final dos modelos ─────────────────────────────────

# 1. Base: dummy e log_reg (de perf_df — secao 5)
perf_cols = ['model', 'accuracy', 'f1', 'roc_auc', 'pr_auc', 'valor_liquido']
base_rows = perf_df[perf_cols].copy()

# 2. Melhor penalizacao (de penalty_df — secao 5c)
best_pen_label = penalty_df['roc_auc_teste'].idxmax()
best_pen = penalty_df.loc[best_pen_label]
pen_row = pd.DataFrame([{
    'model':         f'best_penalty ({best_pen_label}, C={best_pen["melhor_C"]})',
    'accuracy':      best_pen['accuracy_teste'],
    'f1':            best_pen['f1_teste'],
    'roc_auc':       best_pen['roc_auc_teste'],
    'pr_auc':        best_pen['pr_auc_teste'],
    'valor_liquido': best_pen['valor_liquido'],
}])

# 3. Modelo mitigado (de mitig_metrics — secao 7)
mitig_row = pd.DataFrame([{
    'model':         'log_reg_mitigated_equalized_odds',
    'accuracy':      mitig_metrics.get('accuracy', float('nan')),
    'f1':            mitig_metrics.get('f1', float('nan')),
    'roc_auc':       mitig_metrics.get('roc_auc', float('nan')),
    'pr_auc':        mitig_metrics.get('pr_auc', float('nan')),
    'valor_liquido': mitig_metrics.get('valor_liquido', float('nan')),
}])

final_df = pd.concat([base_rows, pen_row, mitig_row], ignore_index=True)

print('Tabela comparativa final:')
display(
    final_df.style
    .format({'accuracy': '{:.4f}', 'f1': '{:.4f}', 'roc_auc': '{:.4f}',
             'pr_auc': '{:.4f}', 'valor_liquido': '{:,.0f}'})
    .highlight_max(subset=['accuracy', 'f1', 'roc_auc', 'pr_auc', 'valor_liquido'], color='#d4edda')
)

# ── Fairness pivot com os 3 modelos (incluindo mitigado) ─────────────────────
print('\nFairness por grupo sensivel:')

# Computa metricas de fairness do modelo mitigado para todos os 4 atributos
sensitive_map_test = {
    'gender':    s_gender_test,
    'age_group': s_age_test,
    'region':    s_region_test,
    'plan_type': s_plan_test,
}
mitig_fair_rows = []
for feat_name, s_feat in sensitive_map_test.items():
    dp = demographic_parity_difference(y_test, y_pred_mitig, sensitive_features=s_feat.values)
    eo = equalized_odds_difference(y_test, y_pred_mitig, sensitive_features=s_feat.values)
    mitig_fair_rows.append({
        'model': 'log_reg_mitigated_equalized_odds',
        'sensitive_feature': feat_name,
        'dp_diff': round(dp, 4),
        'eo_diff': round(eo, 4),
    })

fairness_df_complete = pd.concat(
    [fairness_df[['model', 'sensitive_feature', 'dp_diff', 'eo_diff']],
     pd.DataFrame(mitig_fair_rows)],
    ignore_index=True,
)

pivot = fairness_df_complete.pivot_table(
    index='model', columns='sensitive_feature', values='dp_diff'
).round(4)
display(
    pivot.style
    .format('{:.4f}')
    .highlight_max(color='#ffe0e0')
    .highlight_min(color='#d4edda')
    .set_caption('dp_diff por modelo e atributo sensivel (menor = mais justo)')
)


In [ ]:
# ── Exporta splits preprocessados para reutilização no notebook MLP ─────────
from joblib import dump
import os

os.makedirs('../data/processed', exist_ok=True)

# Extrai o pipeline de preprocessamento já ajustado pelo treino do log_reg
preprocess_fitted = trained["log_reg"].named_steps["prep"]

X_train_arr = preprocess_fitted.transform(X_train).astype("float32")
X_test_arr  = preprocess_fitted.transform(X_test).astype("float32")
y_train_arr = y_train.to_numpy().astype("float32")
y_test_arr  = y_test.to_numpy().astype("float32")

dump(
    (X_train_arr, X_test_arr, y_train_arr, y_test_arr),
    '../data/processed/baseline_splits_arrays.joblib',
)
print(f"[export] X_train: {X_train_arr.shape} | X_test: {X_test_arr.shape}")
print(f"[export] y_train: {y_train_arr.shape} | y_test: {y_test_arr.shape}")
print("[export] Salvo em data/processed/baseline_splits_arrays.joblib")

## 9) Conclusao

---

### 9.1 Desempenho dos modelos

| Modelo | Accuracy | F1 | ROC-AUC | PR-AUC | Valor Liquido |
|--------|----------|----|---------|--------|---------------|
| `dummy_stratified` | 0.5294 | 0.3937 | 0.5046 | 0.3959 | R$ 561.850 |
| `log_reg` (C=1) | 0.8069 | 0.7425 | 0.8783 | 0.8480 | **R$ 1.190.800** |
| `log_reg_best_penalty` (C=0.1) | 0.8069 | 0.7425 | 0.8784 | 0.8480 | R$ 1.190.800 |
| `log_reg_mitigated_equalized_odds` | 0.8007 | 0.7352 | -- | -- | R$ 1.180.950 |

A Regressao Logistica supera o DummyClassifier com ampla margem em todas as metricas. O ROC-AUC de **0.878** confirma boa capacidade discriminativa.

---

### 9.2 Diagnostico de overfitting

| Modelo | delta_roc_auc (treino - teste) | Diagnostico |
|--------|-------------------------------|-------------|
| `log_reg` | **0.0013** | Sem overfitting - modelo generaliza bem |
| `dummy_stratified` | -0.0085 | OK (teste ligeiramente melhor que treino) |

---

### 9.3 Comparacao de penalizacoes (L1 vs L2 vs ElasticNet)

| Penalizacao | Melhor C | ROC-AUC CV | ROC-AUC Teste |
|-------------|----------|-----------|---------------|
| L2 (Ridge)  | 0.1 | 0.8782 | 0.8783 |
| L1 (Lasso)  | 0.1 | 0.8783 | 0.8783 |
| ElasticNet  | 0.1 | 0.8783 | **0.8784** |

As tres penalizacoes convergem para **C = 0.1** com diferenca < 0.0002. **Decisao:** manter L2 com C = 0.1 como referencia pela simplicidade.

---

### 9.4 Fairness por grupo sensivel

#### `log_reg` (sem mitigacao)

| Atributo sensivel | dp_diff | eo_diff |
|-------------------|---------|---------|
| `gender` | 0.0552 | 0.0741 |
| `age_group` | 0.1101 | 0.0758 |
| `region` | 0.1565 | 0.1672 |
| `plan_type` | **0.2960** | **0.1840** |

O maior gap esta em **`plan_type`**: risco regulatorio de concentrar retencao em clientes pre-pagos com menor poder aquisitivo.

#### `log_reg_mitigated_equalized_odds` (mitigacao por `gender`)

| Metrica | `log_reg` | Mitigado | Variacao |
|---------|-----------|----------|----------|
| dp_diff_gender | 0.0552 | **0.0518** | -6% |
| eo_diff_gender | 0.0741 | 0.0847 | +14% |
| F1 | 0.7425 | 0.7352 | -0.007 |
| Valor Liquido | R$ 1.190.800 | R$ 1.180.950 | -R$ 9.850 |

> **Limitacao:** mitigacao treinada apenas com `gender`. Os gaps de `age_group`, `region` e `plan_type` permanecem sem tratamento.

---

### 9.5 Metrica de negocio

```
valor_liquido = TP x R$500 - (TP + FP) x R$50
```

- `log_reg`: **R$ 1.190.800** abordando 3.494 clientes (2.731 TP, 763 FP)
- `dummy_stratified`: 2.254 FP para apenas 1.499 acertos -> R$ 561.850
- Mitigacao custa apenas R$ 9.850 a menos — custo aceitavel para reducao de risco regulatorio

---

### 9.6 Decisao e proximos passos

**Modelo recomendado para a Fase 1:** `log_reg` com **L2, C = 0.1**, threshold padrao 0.5.

**Proximos passos:**

1. **Fairness mais ampla:** mitigacao simultanea em `plan_type` e `region`
2. **Modelos nao-lineares:** Random Forest e XGBoost para comparar com Regressao Logistica
3. **Threshold otimizado:** calibrar para maximizar `valor_liquido`
4. **Diagnostico de coeficientes:** analise VIF e importancia de features
5. **Persistencia do modelo:** serializar o pipeline para servir predicoes em producao
